# Week 2 · Custom Knowledge: RAG, KAG & GraphRAG
### Build Custom AI — SarasAI Live Session 2

Last week our VLM produced defect notes. But when someone asks *"have customers complained
about this kind of crack before?"* — the model has no idea. Its weights were frozen months ago
and have never seen **your** data.

**Retrieval-Augmented Generation (RAG)** fixes this without any training: fetch the relevant
internal documents at question time and hand them to the model as context.

**What you will do today**

1. Understand embeddings as a *search technology* (building on Week 1's geometry intuition).
2. Build a complete RAG pipeline: chunk → embed → index → retrieve → generate with citations.
3. See where flat RAG breaks — and how **GraphRAG/KAG** handle relational questions.
4. Evaluate properly: **retrieval recall@k** and **faithfulness** on a frozen gold Q&A set.

> 📏 **The Evaluation Rule, week 2 edition:** a RAG system has *two* independent failure points —
> retrieval and generation. We will measure them **separately**.

> ✍️ **LIVE-CODING WORKBOOK** — same notebook as `week2_rag_kag_graphrag.ipynb`, with the teaching-core cells left as `# TODO (live)` skeletons we write together in the session. Boilerplate (installs, corpus, model loads, gold set) is pre-filled. The fully-coded notebook is the answer key — keep it open next to this one.

---
## 0 · Setup

`sentence-transformers` for embeddings, `faiss-cpu` for the vector index, `transformers` for
the generator LLM. *(FAISS = Facebook AI Similarity Search — the workhorse in-memory vector store.)*

In [ ]:
!pip install -q sentence-transformers faiss-cpu "transformers>=4.50" datasets accelerate

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
# T4s (pre-Ampere) have no native bfloat16 — detect and fall back to float16
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print("Using dtype:", DTYPE)

---
## 1 · The knowledge base: product reviews + tech specs

Continuing the capstone scenario: our company sells kitchen appliances. Two knowledge sources:

- **Customer reviews** — the voice of the field. We pull real ones from
  `McAuley-Lab/Amazon-Reviews-2023` (Appliances category).
- **Tech specs / internal docs** — we define a small set inline so the corpus has both
  *observed problems* (reviews) and *ground truth about the product* (specs).

In your company these would be support tickets, wikis, PDFs — the pipeline is identical.

In [ ]:
from datasets import load_dataset

# NOTE: this dataset uses a legacy loading script that modern `datasets` refuses to run —
# so we stream the raw JSONL file directly off the Hub. Same data, no script needed.
reviews_raw = load_dataset(
    "json",
    data_files="hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/Appliances.jsonl",
    split="train",
    streaming=True,               # the full dataset is huge — stream and take what we need
)

reviews = []
for r in reviews_raw:
    if r["text"] and 200 < len(r["text"]) < 1200:      # substantial but not rambling
        reviews.append({"source": f"review_{len(reviews):03d}", "text": r["text"].strip()})
    if len(reviews) >= 120:
        break

print(f"Collected {len(reviews)} reviews")
print("\nSample:", reviews[0]["text"][:200], "…")

And a handful of internal spec documents — deliberately containing facts the reviews
*don't* have (warranty terms, materials, part numbers), so we can later ask questions that
require both sources:

In [ ]:
specs = [
    {"source": "spec_kettle",  "text": "Model KT-2000 electric kettle. Capacity 1.7L. Body: borosilicate "
        "glass with a polypropylene lid assembly. Lid hinge part number PP-114. Boil-dry protection "
        "included. Warranty: 24 months covering material and manufacturing defects."},
    {"source": "spec_toaster", "text": "Model TS-400 two-slot toaster. Stainless steel housing. Crumb tray "
        "part number CT-security-09. Known service note: units manufactured before 2023-06 may exhibit "
        "lever sticking; resolved by revision C of the spring assembly, part SP-77C."},
    {"source": "spec_blender", "text": "Model BL-900 blender. 900W motor, tritan jar 1.5L. Blade assembly "
        "part number BA-45. The jar is NOT dishwasher safe above 60°C. Warranty: 12 months, motor only."},
    {"source": "policy_returns", "text": "Return policy for all products (models KT-2000, TS-400, BL-900): "
        "defective units can be returned within 30 days for a full refund. Warranty claims after 30 days "
        "are handled as repair or replacement. Water damage and misuse are excluded from warranty coverage."},
]

corpus = reviews + specs
print(f"Total corpus: {len(corpus)} documents")

---
## 2 · Chunking: the unglamorous step that decides RAG quality

Embedding models have a sweet spot (~100–300 words). Too-long chunks blur multiple topics into
one vector; too-short chunks lose context. Our documents are already short, so we use a simple
word-window splitter with **overlap** — overlap prevents a sentence being cut in half at a
boundary and lost to search.

*(In production you'd use structure-aware splitting — by section, paragraph, or heading. The
principle is the same.)*

In [ ]:
def chunk(text, source, max_words=120, overlap=30):
    """Split text into overlapping word-window chunks tagged with their source id."""
    # TODO (live): word-window chunking —
    #   words = text.split(); step = max_words - overlap
    #   slide over words in steps, keep {"source": source, "text": " ".join(window)}
    ...

---
## 3 · Embeddings: from Week 1's intuition to a search engine

Last week we saw that `crack` and `fracture` live near each other in embedding space.
A **sentence embedding model** does the same for whole passages — trained so that a question
and the passage answering it land close together.

We use `BAAI/bge-small-en-v1.5`: 33M parameters, excellent quality-per-FLOP, runs fine on CPU.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

# TODO (live): embed every chunk text, L2-normalized (normalize_embeddings=True), float32
# -> doc_vecs  (why normalized? inner product == cosine — next cell depends on it)
...

Every chunk is now a point in 384-dimensional space. Searching = *"embed the question,
find the nearest points."* FAISS does that lookup efficiently:

In [ ]:
import faiss

# BGE retrieves better when the QUERY carries this prefix (documents stay bare):
BGE_PREFIX = "Represent this sentence for searching relevant passages: "

# TODO (live): 1) build faiss.IndexFlatIP(dim) and add doc_vecs
#              2) retrieve(question, k=4): encode BGE_PREFIX+question -> index.search
#                 -> [{**chunks[i], "score": float(s)}, ...]
...

Look at what came back: semantically related passages, even where the *words* differ
("broke off" vs "snapped", "hinge"). That is the whole magic of dense retrieval — and note
that **no LLM has been involved yet**. Retrieval is cheap, fast, and independently testable.

---
## 4 · The G in RAG: grounded generation

Now we assemble the classic RAG prompt: retrieved chunks + rules + question. The rules matter
more than the plumbing —

1. **Answer only from the context** — this is the anti-hallucination clause.
2. **Cite sources** — so every claim is checkable.
3. **Say "I don't know" when the context doesn't contain the answer** — a RAG system that
   can't say this is a liability, not an asset.

Generator: `Qwen2.5-1.5B-Instruct` — small, fast, T4-friendly. RAG quality is usually
retrieval-bound, not generator-bound, so a small model goes surprisingly far.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_ID = "Qwen/Qwen2.5-1.5B-Instruct"
gen_tok = AutoTokenizer.from_pretrained(GEN_ID)
gen_lm = AutoModelForCausalLM.from_pretrained(
    GEN_ID, torch_dtype=DTYPE, device_map="auto"
)
print(f"Generator loaded: {gen_lm.num_parameters()/1e9:.2f}B params")

In [ ]:
def generate(messages, max_new_tokens=300):
    """Plain chat helper around the generator LLM."""
    inputs = gen_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(gen_lm.device)
    out = gen_lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=gen_tok.eos_token_id)
    return gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
def rag_answer(question, k=4):
    """Grounded, cited answer + the hits used."""
    # TODO (live): the three-block grounded prompt —
    #   CONTEXT: "[source] text" lines from retrieve()
    #   RULES:   answer ONLY from context · cite [ids] · refuse with
    #            "I don't have enough information" when context doesn't cover it
    #   QUESTION last -> generate(), return (answer, hits)
    ...

Test drive — one question answerable from specs, one from reviews, one answerable from
**neither** (the honesty test):

In [ ]:
for q in [
    "What is the warranty on the KT-2000 kettle?",
    "What problems do customers report most often?",
    "What is the CEO's email address?",               # not in the corpus — should refuse
]:
    answer, _ = rag_answer(q)
    print(f"Q: {q}\nA: {answer}\n" + "-" * 70)

---
## 5 · Why RAG matters: the with/without comparison

Same model, same question — with retrieval vs. without. This single comparison is your
strongest argument when someone asks "why do we need all this infrastructure?":

In [ ]:
q = "Is the BL-900 blender jar dishwasher safe?"

bare = generate([{"role": "user", "content": q}])
grounded, _ = rag_answer(q)

print("WITHOUT retrieval:\n", bare[:300])
print("\nWITH retrieval:\n", grounded)

Without retrieval, the model *must* guess (and will do so fluently — that's the danger).
With retrieval it states the spec-sheet fact and cites it.

> **Fine-tuning would not fix the bare model.** Facts belong in retrieval; *behavior* belongs
> in fine-tuning — that's next week. This decision boundary is the most important slide of
> the whole course.

---
## 6 · Where flat RAG breaks: relational questions & GraphRAG

Try a question whose answer is spread across **multiple documents linked by an entity**:

*"Which part number should be replaced for toasters with sticking levers, and does the return
policy cover a unit bought 3 months ago?"*

Answering needs: the toaster spec (part SP-77C) **+** the returns policy (30-day / warranty
distinction). Flat top-k retrieval often gets one but not both — nothing in the *embedding* of
the returns policy mentions toasters.

**GraphRAG** (Edge et al. 2024, arXiv:2404.16130) addresses this by extracting an
**entity graph** from the corpus first, then retrieving along edges. Let's build a miniature
version to make the idea concrete — we extract entities with our LLM, link chunks that share
them:

In [ ]:
import json, re

def extract_entities(text):
    """LLM-based entity extraction — the first step of any GraphRAG pipeline."""
    prompt = (
        "List the product models, part numbers, and defect types mentioned in this text. "
        'Respond ONLY with JSON: {"entities": ["...", "..."]}\n\nText: ' + text
    )
    raw = generate([{"role": "user", "content": prompt}], max_new_tokens=100)
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    try:
        return [str(e).lower() for e in json.loads(m.group())["entities"]] if m else []
    except Exception:            # best-effort extractor: any malformed output → no entities
        return []

# extract for the spec/policy docs (small corpus → run on specs only, to keep the demo fast)
graph = {}                       # entity -> set of source ids
for doc in specs:
    for ent in extract_entities(doc["text"]):
        graph.setdefault(ent, set()).add(doc["source"])

for ent, sources in list(graph.items())[:8]:
    print(f"{ent:30} → {sorted(sources)}")

Now **graph-augmented retrieval**: embed-retrieve as before, then *expand* — pull in every
document that shares an entity with what we found. One hop along the graph:

In [ ]:
def graph_retrieve(question, k=3):
    """Vector retrieval + 1-hop entity-graph expansion."""
    # TODO (live): 1) hits = retrieve(question, k)
    #              2) collect entities of hit sources from the graph
    #              3) pull in any OTHER doc sharing an entity (the 1-hop expansion)
    ...

If the extractor did its job, the graph hop pulls in `policy_returns` even when its
embedding is nowhere near "toaster lever" — because entity linkage (the TS-400 appears in both
documents), not similarity, connects them. *(With a 1.5B extractor this can occasionally miss —
which is itself the lesson: GraphRAG quality is bounded by extraction quality, and production
systems use a strong model for the indexing pass.)*

**Terminology map** for your reading:
- **RAG** — retrieve by vector similarity. Best default; cheapest.
- **GraphRAG** — retrieve via an entity/community graph built offline. Wins on multi-hop and
  "summarize the whole corpus" questions; costs an expensive indexing phase.
- **KAG** — Knowledge-Augmented Generation: umbrella term for wiring *structured* knowledge
  (KGs, databases, rules) into generation, of which GraphRAG is one recipe.

Rule of thumb: **start flat**. Add graph structure only when your eval (next section!) shows
multi-hop questions failing.

---
## 7 · Evaluation: the frozen gold Q&A set

Now the week's discipline. We author a **gold set** — question, expected source(s), reference
answer — *by reading the corpus*, then freeze it. Two separate scores:

1. **Retrieval recall@k** — for what fraction of questions did the top-k chunks include the
   gold source? *(No LLM involved — pure search quality.)*
2. **Faithfulness** — is every claim in the generated answer supported by the retrieved
   context? We use an **LLM-as-judge** here, which is exactly how RAGAS
   (Es et al. 2023, arXiv:2309.15217) automates it at scale.

In [ ]:
gold = [
    {"q": "What is the warranty period for the KT-2000 kettle?",
     "sources": ["spec_kettle"],  "ref": "24 months"},
    {"q": "What part number is the kettle lid hinge?",
     "sources": ["spec_kettle"],  "ref": "PP-114"},
    {"q": "Is the blender jar dishwasher safe?",
     "sources": ["spec_blender"], "ref": "60 degrees is the limit"},
    {"q": "Which spring assembly revision fixed the toaster lever sticking?",
     "sources": ["spec_toaster"], "ref": "revision C, part SP-77C"},
    {"q": "How long do customers have to return a defective unit for a refund?",
     "sources": ["policy_returns"], "ref": "30 days"},
    {"q": "Does the warranty cover water damage?",
     "sources": ["policy_returns"], "ref": "excluded from coverage"},
]
print(f"Gold set: {len(gold)} questions — FROZEN 🔒 (in the capstone: ≥20, written before building)")

In [ ]:
def recall_at_k(gold_set, k=4):
    """Fraction of gold questions whose required sources ALL appear in top-k."""
    # TODO (live): set(item["sources"]) <= retrieved ids -> count / len(gold_set)
    ...

for k in [1, 2, 4, 8]:
    print(f"recall@{k}: {recall_at_k(gold, k):.0%}")

Read this table the way an engineer reads it: if recall@8 is high but recall@2 is low, your
*embeddings are fine* but you need a **reranker** or better chunking. If even recall@8 is low,
no prompt engineering downstream can save you — **the generator can't cite what retrieval
never fetched.**

Now faithfulness — the LLM-as-judge pattern:

In [ ]:
def judge_faithfulness(question, answer, context):
    """1 if every claim in answer is supported by context, else 0 — LLM-as-judge."""
    # TODO (live): one-word YES/NO judge prompt -> generate(max_new_tokens=5) -> parse
    ...

In [ ]:
def judge_relevance(question, answer):
    """Answer relevance — does the answer actually address the question asked?"""
    prompt = (f"Question: {question}\nAnswer: {answer}\n\n"
              "Does the Answer directly address the Question (regardless of whether it is "
              "factually correct)? Reply with exactly one word: YES or NO.")
    verdict = generate([{"role": "user", "content": prompt}], max_new_tokens=5)
    return 1 if "YES" in verdict.upper() else 0

Faithfulness and relevance are *independent axes* — an answer can be perfectly grounded in
the context yet not answer the question (faithful, irrelevant), or spot-on topic but invented
(relevant, unfaithful). RAGAS scores both; so do we:

In [ ]:
faithful = relevant = correct_hint = 0
for item in gold:
    answer, hits = rag_answer(item["q"])
    context = "\n".join(h["text"] for h in hits)
    faithful += judge_faithfulness(item["q"], answer, context)
    relevant += judge_relevance(item["q"], answer)
    correct_hint += int(item["ref"].lower().split()[0] in answer.lower())  # crude reference check

print(f"Faithfulness:      {faithful}/{len(gold)}")
print(f"Answer relevance:  {relevant}/{len(gold)}")
print(f"Contains reference answer (crude): {correct_hint}/{len(gold)}")

> ⚠️ **Honest caveats, so you use this correctly:**
> 1. A 1.5B judge is a *smoke detector*, not an auditor. For the capstone, use the `ragas`
>    library with a stronger judge model — same idea, calibrated implementation.
> 2. Judge and generator sharing weights (as here, for VRAM reasons) risks self-agreement bias.
>    In real evals, **judge ≠ generator**.
> 3. Our "contains reference" check is deliberately crude — it catches gross failures only.
>    Rubric-based scoring arrives in Week 3.

---
## 8 · Production considerations: RAG evaluation & monitoring

The two-failure-mode split you just measured **is** the production monitoring design:

| What can silently degrade | Detection |
|---|---|
| New docs added, index stale | **Version the index**; re-embed on doc changes |
| Corpus grows → retrieval quality drifts | recall@k on the gold set, re-run **on every index build** |
| Model/prompt updated → faithfulness regresses | faithfulness suite in CI, alert on drop |
| Users ask questions nobody anticipated | log queries + retrieval scores; low-score queries → review queue → new gold items |

The gold set is a *living asset*: every real failure becomes a new row. Teams that do this have
RAG that improves monthly; teams that don't have RAG that quietly rots.

---
## 9 · Your assignments

**Ungraded warm-up:** add 30 more reviews to the corpus, re-embed, and check whether
recall@4 moved. (It can go *down* — more haystack, same needles. Understanding why is the point.)

**Graded — Capstone Increment 2:** index the capstone reviews + spec corpus and deliver the RAG
query interface. Submit: (1) the notebook, (2) your frozen gold set (≥20 questions, authored
before building), (3) recall@k table for k∈{1,2,4,8}, (4) faithfulness **and answer-relevance**
scores — use the `ragas` library with a stronger judge model than your generator, (5) five
example answers with citations.

**Read before next week** *(≈45 min)*:
- LoRA — Hu et al. 2021, [arXiv:2106.09685](https://arxiv.org/abs/2106.09685) (§1–3)
- QLoRA — Dettmers et al. 2023, [arXiv:2305.14314](https://arxiv.org/abs/2305.14314) (skim §3)

*Next week: RAG gave the model your knowledge. Fine-tuning gives it your **behavior** —
we train the Analyst SLM.*